# Set up

In [2]:
import duckdb
con = duckdb.connect("../data/yelp.duckdb")

In [3]:
con.sql("SHOW TABLES").df()

,name
0,business_rag
1,business_raw
2,business_restaurants
3,checkin_rag
4,review_rag
5,tip_rag
6,user_rag


# Load reviews

In [4]:
con.sql("""
CREATE OR REPLACE TABLE review_rag AS
SELECT review_id, business_id, user_id, stars, date, text
FROM read_json_auto(
    '../data/yelp_academic_dataset_review.json',
    format='newline_delimited'
)
WHERE business_id IN (SELECT business_id FROM business_rag)""")

con.sql("SELECT COUNT(*) AS reviews FROM review_rag").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,reviews
0,476421


In [5]:
con.sql("""
SELECT
  SUM(CASE WHEN actual = review_count THEN 1 ELSE 0 END) AS exact_match,
  SUM(CASE WHEN actual > review_count THEN 1 ELSE 0 END) AS more_reviews_than_stated,
  SUM(CASE WHEN actual < review_count THEN 1 ELSE 0 END) AS fewer_reviews_than_stated
FROM (
    SELECT b.business_id, b.review_count, COUNT(r.review_id) AS actual
    FROM business_rag b
    LEFT JOIN review_rag r USING (business_id)
    GROUP BY b.business_id, b.review_count)""").df()

,exact_match,more_reviews_than_stated,fewer_reviews_than_stated
0,807.0,1452.0,0.0


The review file has 476,421 reviews for the New Orleans restaurants, and the business file’s review_count adds up to 465,982. I compared them for each restaurant: 807 match exactly, 1,452 have more reviews in the review file, and none have fewer. So no reviews were lost when loading. I use the real row count (476,421) in all later work, and treat review_count as a rough guide.

In [6]:
con.sql("""
CREATE OR REPLACE TABLE review_stats AS
SELECT review_id, business_id, user_id, stars, date
FROM read_json_auto(
    '../data/yelp_academic_dataset_review.json',
    format='newline_delimited'
)
WHERE business_id IN (SELECT business_id FROM business_restaurants)""")

con.sql("SELECT COUNT(*) AS reviews FROM review_stats").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,reviews
0,4724471


# Check the reviews

In [7]:
# duplicate review ids (expect 0)
con.sql("SELECT COUNT(*) - COUNT(DISTINCT review_id) AS duplicate_ids FROM review_rag").df()

,duplicate_ids
0,0


In [8]:
# missing values
con.sql("""
SELECT COUNT(*) AS total,
       COUNT(*) - COUNT(text)    AS missing_text,
       COUNT(*) - COUNT(stars)   AS missing_stars,
       COUNT(*) - COUNT(user_id) AS missing_user
FROM review_rag""").df()

,total,missing_text,missing_stars,missing_user
0,476421,0,0,0


In [9]:
# date range and star distribution
con.sql("SELECT MIN(date) AS first_review, MAX(date) AS last_review FROM review_rag").df()

,first_review,last_review
0,2005-03-14 18:07:51,2022-01-19 19:47:59


In [10]:
con.sql("SELECT stars, COUNT(*) AS reviews FROM review_rag GROUP BY stars ORDER BY stars").df()

,stars,reviews
0,1.0,42639
1,2.0,35258
2,3.0,53906
3,4.0,118198
4,5.0,226420


In [11]:
con.sql("""
SELECT MIN(LENGTH(text)) AS min_chars,
       ROUND(AVG(LENGTH(text))) AS avg_chars,
       MAX(LENGTH(text)) AS max_chars
FROM review_rag""").df()

,min_chars,avg_chars,max_chars
0,1,536.0,5000


In [12]:
con.sql("""
SELECT COUNT(*) AS restaurants_with_reviews
FROM business_rag
WHERE business_id IN (SELECT DISTINCT business_id FROM review_rag)""").df()

,restaurants_with_reviews
0,2259


In [13]:
con.sql("""
SELECT COUNT(*) AS total,
       COUNT(*) - COUNT(DISTINCT review_id) AS duplicate_ids,
       COUNT(*) - COUNT(stars)              AS missing_stars,
       COUNT(*) - COUNT(date)               AS missing_date,
       COUNT(*) - COUNT(user_id)            AS missing_user
FROM review_stats""").df()

,total,duplicate_ids,missing_stars,missing_date,missing_user
0,4724471,0,0,0,0


In [14]:
con.sql("SELECT MIN(date) AS first_review, MAX(date) AS last_review FROM review_stats").df()

,first_review,last_review
0,2005-02-16 03:23:22,2022-01-19 19:48:25


In [15]:
con.sql("SELECT stars, COUNT(*) AS reviews FROM review_stats GROUP BY stars ORDER BY stars").df()

,stars,reviews
0,1.0,567185
1,2.0,404486
2,3.0,543108
3,4.0,1130251
4,5.0,2079441


In [16]:
con.sql("""
SELECT COUNT(*) AS new_orleans_reviews
FROM review_stats
WHERE business_id IN (SELECT business_id FROM business_rag)""").df()

,new_orleans_reviews
0,476421


All 2,259 restaurants have at least one review in the review file, so no restaurant is left without text for the RAG system.

# Load tips and check-ins

In [17]:
con.sql("""
CREATE OR REPLACE TABLE tip_rag AS
SELECT business_id, user_id, date, text 
FROM read_json_auto('../data/yelp_academic_dataset_tip.json', format='newline_delimited')
WHERE business_id IN (SELECT business_id FROM business_rag)""")

con.sql("""
CREATE OR REPLACE TABLE checkin_rag AS
SELECT * FROM read_json_auto('../data/yelp_academic_dataset_checkin.json', format='newline_delimited')
WHERE business_id IN (SELECT business_id FROM business_rag)""")

con.sql("""
SELECT 'tip_rag' AS table_name, COUNT(*) AS rows FROM tip_rag
UNION ALL
SELECT 'checkin_rag', COUNT(*) FROM checkin_rag""").df()

,table_name,rows
0,tip_rag,62945
1,checkin_rag,2232


# Load users

In [18]:
con.sql("""
CREATE OR REPLACE TABLE user_rag AS
SELECT user_id, name, review_count, yelping_since, average_stars 
FROM read_json_auto('../data/yelp_academic_dataset_user.json', format='newline_delimited')
WHERE user_id IN (SELECT DISTINCT user_id FROM review_rag)""")

con.sql("SELECT COUNT(*) AS users FROM user_rag").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,users
0,201798


In [19]:
con.sql("""
CREATE OR REPLACE TABLE user_stats AS
SELECT user_id, review_count
FROM read_json_auto(
    '../data/yelp_academic_dataset_user.json',
    format='newline_delimited'
)
WHERE user_id IN (SELECT DISTINCT user_id FROM review_stats)""")

con.sql("SELECT COUNT(*) AS users FROM user_stats").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,users
0,1445984


# Check that the tables join

In [20]:
con.sql("""
SELECT
  SUM(CASE WHEN business_id NOT IN (SELECT business_id FROM business_rag) THEN 1 ELSE 0 END) AS reviews_without_business,
  SUM(CASE WHEN user_id NOT IN (SELECT user_id FROM user_rag) THEN 1 ELSE 0 END) AS reviews_without_user
FROM review_rag""").df()

,reviews_without_business,reviews_without_user
0,0.0,0.0


In [21]:
con.sql("""
SELECT COUNT(*) AS reviews_without_user
FROM review_stats
WHERE user_id NOT IN (SELECT user_id FROM user_stats)""").df()

,reviews_without_user
0,7


In [22]:
# summary of all scoped tables
con.sql("""
SELECT 'business_rag' AS table_name, COUNT(*) AS rows FROM business_rag
UNION ALL SELECT 'review_rag', COUNT(*) FROM review_rag
UNION ALL SELECT 'tip_rag', COUNT(*) FROM tip_rag
UNION ALL SELECT 'checkin_rag', COUNT(*) FROM checkin_rag
UNION ALL SELECT 'user_rag', COUNT(*) FROM user_rag""").df()

,table_name,rows
0,business_rag,2259
1,review_rag,476421
2,tip_rag,62945
3,checkin_rag,2232
4,user_rag,201798


In [23]:
con.sql("""
SELECT 'business_restaurants' AS table_name, COUNT(*) AS rows FROM business_restaurants
UNION ALL SELECT 'review_stats', COUNT(*) FROM review_stats
UNION ALL SELECT 'user_stats', COUNT(*) FROM user_stats""").df()

,table_name,rows
0,business_restaurants,52268
1,review_stats,4724471
2,user_stats,1445984


# Conclusion

## What I loaded

**New Orleans tables (with review text, for the RAG system)**

| Table | Rows |
|---|---|
| business_rag | 2,259 |
| review_rag | 476,421 |
| tip_rag | 62,945 |
| checkin_rag | 2,232 |
| user_rag | 201,798 |

**All-states tables (no review text, for the SQL analysis and dashboard)**

| Table | Rows |
|---|---|
| business_restaurants | 52,268 |
| review_stats | 4,724,471 |
| user_stats | 1,445,984 |

All tables keep only the columns I need. The New Orleans tables cover restaurants in New Orleans and the users who reviewed them. The all-states tables cover every restaurant in the dataset.

## Checks
- In the New Orleans tables, every review has a matching restaurant and user (0 without).
- The review file has 476,421 reviews for New Orleans, 2.2% more than the business file's `review_count` (465,982). No restaurant has fewer reviews than stated, so nothing was lost.
- All 2,259 New Orleans restaurants have at least one review.
- 27 New Orleans restaurants have no check-in row. Check-ins are not used for the RAG system, so this is fine.
- New Orleans reviews run from 2005-03-14 to 2022-01-19. The average review is 536 characters long.
- `review_stats` runs from 2005-02-16 to 2022-01-19. The first and last years are partial, so I will take care when comparing years.

In [24]:
con.close()